<a href="https://colab.research.google.com/github/pantelis/eng-ai-agents/blob/main/notebooks/VLM/clip/clip_zero_shot.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Zero-shot classification as a linear classifier (CLIP)

This notebook demonstrates that zero-shot classification with CLIP can be interpreted as a **linear classifier whose weights are generated from text prompts**. We follow the notation used on the [CLIP lecture](/aiml-common/lectures/VLM/clip/index) page: $\boldsymbol\ell$ for images, $\mathbf{t}$ for text, $f_\ell$ and $f_t$ for the two encoders, and $\mathbf{z}_\ell = f_\ell(\boldsymbol\ell)$, $\mathbf{z}_t = f_t(\mathbf{t})$ for the embeddings.

You will:
1. Derive the formulation
2. Implement zero-shot classification
3. Visualize the text classifier weights and the image in the shared embedding space

## 1. Derivation

CLIP learns two encoders (see the [CLIP lecture](/aiml-common/lectures/VLM/clip/index) for the contrastive training objective):

$$
\mathbf{z}_\ell = f_\ell(\boldsymbol\ell), \quad \mathbf{z}_t = f_t(\mathbf{t})
$$

where $\boldsymbol\ell$ is an image, $\mathbf{t}$ is a text string, $f_\ell$ is the image encoder, and $f_t$ is the text encoder. Both output unit vectors on the surface of the hypersphere $\mathbb{S}^{d_z}$.

For zero-shot classification over classes $y \in \{a, b, c, \ldots\}$, we write a prompt $\mathbf{t}^y$ for each class and embed it:

$$
\mathbf{z}_t^y = f_t(\mathbf{t}^y)
$$

Given a query image $\boldsymbol\ell^q$ with embedding $\mathbf{z}_\ell^q = f_\ell(\boldsymbol\ell^q)$, the prediction is the class whose text embedding has the largest dot product with the image embedding:

$$
\hat{y} = \arg\max_y \; (\mathbf{z}_\ell^q)^\top \mathbf{z}_t^y
$$

This is exactly a **linear classifier** over $\mathbf{z}_\ell^q$ whose weight vectors are the per-class text embeddings:

$$
\hat{y} = \arg\max_y \; (\mathbf{z}_t^y)^\top \mathbf{z}_\ell^q
$$

> Each text prompt $\mathbf{t}^y$ instantiates one classifier weight vector $\mathbf{z}_t^y$. There is no training — the classifier is built on the fly from language.

In [ ]:
!pip install transformers torch torchvision pillow matplotlib scikit-learn

In [ ]:
import torch
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

## 2. Load a query image $\boldsymbol\ell^q$

In [ ]:
url = "https://images.unsplash.com/photo-1518717758536-85ae29035b6d"
image = Image.open(requests.get(url, stream=True).raw)
plt.imshow(image)
plt.axis("off")

## 3. Define prompts $\mathbf{t}^y$ (classifier weights)

In [ ]:
labels = ["a dog", "a cat", "a car", "a plane"]
prompts = [f"a photo of {label}" for label in labels]

## 4. Compute the embeddings $\mathbf{z}_\ell^q$ and $\{\mathbf{z}_t^y\}$

In [ ]:
inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

# Normalize (important for cosine similarity)
image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

## 5. Zero-shot classification

Apply the classification rule from Section 1:

$$
\hat{y} = \arg\max_y \; (\mathbf{z}_\ell^q)^\top \mathbf{z}_t^y
$$

### Why raw softmax looks flat

CLIP cosine similarities live in a narrow range. Both encoders produce $L_2$-normalized embeddings on $\mathbb{S}^{d_z}$, and in practice the joint space is a tight cone, so raw dot products $(\mathbf{z}_\ell^q)^\top \mathbf{z}_t^y$ typically sit between **0.18 and 0.30** even for good matches. A softmax applied directly to such close values produces a nearly uniform distribution, which makes the classifier look much less confident than it actually is.

The fix is the learned **temperature** parameter $\tau$ that CLIP was trained with (the same $\tau$ that appears in the InfoNCE loss on the [CLIP lecture](/aiml-common/lectures/VLM/clip/index) page). In Stable-Baselines3's HuggingFace wrapper this is stored as `logit_scale = 1/τ ≈ 100`. Inference must scale the similarities by $1/\tau$ before the softmax:

$$
p(y \mid \boldsymbol\ell^q) = \mathrm{softmax}_y\!\left(\frac{1}{\tau} \cdot (\mathbf{z}_\ell^q)^\top \mathbf{z}_t^y\right)
$$

HuggingFace's CLIP model exposes this scaled value as `outputs.logits_per_image`, so you get the sharp classification output for free. The cell below compares all three: raw cosine similarity, naive softmax (wrong), temperature-scaled softmax (correct), and the HuggingFace convenience.

In [ ]:
# Raw cosine similarity between image and each text prompt
similarity = (image_embeds @ text_embeds.T).squeeze(0)

# 1. Naive softmax — this is the common pitfall
naive_probs = similarity.softmax(dim=0)

# 2. Correctly scaled softmax using CLIP's learned temperature
#    `logit_scale` is a learned parameter (≈4.6 in log space, ≈100 in linear),
#    and CLIP was trained with contrastive loss at that scale.
logit_scale = model.logit_scale.exp().item()
scaled_probs = (similarity * logit_scale).softmax(dim=0)

# 3. HuggingFace convenience — logits_per_image already has the scale applied
hf_probs = outputs.logits_per_image.softmax(dim=-1).squeeze(0)

print(f"Learned logit_scale (temperature): {logit_scale:.2f}")
print()
print(f"{'label':<10} {'cos sim':>10} {'naive':>10} {'scaled':>10} {'HF':>10}")
print("-" * 55)
for i, label in enumerate(labels):
    print(
        f"{label:<10} "
        f"{similarity[i].item():>10.4f} "
        f"{naive_probs[i].item():>10.4f} "
        f"{scaled_probs[i].item():>10.4f} "
        f"{hf_probs[i].item():>10.4f}"
    )

## 6. Visualizing the classifier weights and the image in the same space

Plotting the first 50 components of each text embedding side by side is not very informative — the embeddings live on a $d_z$-dimensional unit hypersphere and the raw coordinates have no intrinsic meaning. A better view is to project the weights into a low-dimensional subspace and see how they are laid out relative to each other and to the image embedding $\mathbf{z}_\ell^q$.

Below we fit a 2-component PCA to the text embeddings $\{\mathbf{z}_t^y\}$ and project both the text prompts and the image into the same plane. Each labeled point is one classifier weight; the red star is the image embedding $\mathbf{z}_\ell^q$. The closest label to the star is the zero-shot prediction $\hat{y}$.

In [ ]:
from sklearn.decomposition import PCA

# Project the text classifier weights to 2D with PCA so we can see them
# arranged in the shared CLIP embedding space. Each point is one prompt,
# and semantically related prompts should cluster together.
weights = text_embeds.numpy()          # {z_t^y}
image_vec = image_embeds.numpy()       # z_l^q

pca = PCA(n_components=2)
pca.fit(weights)

proj_text = pca.transform(weights)
proj_image = pca.transform(image_vec)

fig, ax = plt.subplots(figsize=(7, 6))

# Text classifier weights z_t^y
ax.scatter(proj_text[:, 0], proj_text[:, 1], s=180, c="steelblue", edgecolors="black", zorder=3, label=r"text prompts $z_t^y$ (classifier weights)")
for i, label in enumerate(labels):
    ax.annotate(
        label,
        (proj_text[i, 0], proj_text[i, 1]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=11,
    )

# Image embedding z_l^q in the same PCA basis
ax.scatter(proj_image[:, 0], proj_image[:, 1], s=260, c="crimson", marker="*", edgecolors="black", zorder=4, label=r"image embedding $z_\ell^q$")

ax.set_xlabel(f"PC1  ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2  ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("Classifier weights and query image in 2D (PCA of text embeddings)")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPCA variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
print()
# Temperature-scaled logits and softmax probabilities — consistent with Section 5
logit_scale = model.logit_scale.exp().item()
raw_sims = (image_vec @ weights.T).flatten()
logits = logit_scale * raw_sims
probs = np.exp(logits - logits.max())
probs = probs / probs.sum()

print(f"Temperature-scaled classification (1/tau = {logit_scale:.2f}):")
print(f"{'label':<10} {'cos sim':>10} {'logit':>10} {'prob':>10}")
print("-" * 45)
for label, s, lg, p in zip(labels, raw_sims, logits, probs):
    print(f"{label:<10} {s:>10.4f} {lg:>10.2f} {p:>10.4f}")
print()
print(f"Predicted class: {labels[int(np.argmax(probs))]}")

## 7. Conclusion

We demonstrated that:

- Each class prompt $\mathbf{t}^y$ produces a vector $\mathbf{z}_t^y = f_t(\mathbf{t}^y)$
- These vectors act as classifier weights
- Zero-shot classification is linear classification in the shared CLIP embedding space

$$
\hat{y} = \arg\max_y \; (\mathbf{z}_t^y)^\top \mathbf{z}_\ell^q
$$

> Language defines the classifier.